# 03 | Feminine Hygiene — Recovery Classification

Compare five classifiers and evaluate the selected distance-weighted KNN model. Category-specific K thresholds and feature sets are defined in `src/model_config.py`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from IPython.display import display
from src.modeling import compare_models, forward_selection, final_knn, exploratory_revenue_proxy
category = 'feminine_hygiene'
input_path = ROOT / 'data/private/feminine_hygiene_events.xlsx'
df = pd.read_excel(input_path, sheet_name='promotion_events_K') if input_path.exists() else None


## Model comparison

The original design uses an 80/20 stratified split, feature scaling and class/sample weighting where supported. KNN uses distance-weighted voting.

In [ ]:
if df is not None:
    display(compare_models(df, category))
else:
    print('Add the authorized local event table to run this section.')

## Forward feature selection

Evaluate candidate features sequentially, retaining additions that improve macro F1 on the fixed evaluation split.

In [ ]:
if df is not None:
    display(forward_selection(df, category))

## Selected KNN and feature diagnostics

The final feature set is fixed to the configuration used in the original selected-model notebook.

In [ ]:
if df is not None:
    metrics, cm, diagnostics = final_knn(df, category)
    display(pd.Series(metrics))
    display(pd.DataFrame(cm, index=['Medium','Low','High'], columns=['Medium','Low','High']))
    display(diagnostics)

## Reported classification results

In [ ]:
display(pd.read_csv(ROOT / 'results/reported_model_results.csv').query('category == "Feminine hygiene"'))

## Exploratory revenue-proxy regression

The two-stage approach supplies K-class information to an XGBoost regressor. The target uses a baseline revenue estimate when available and a historical proxy otherwise. See [methodology](../docs/methodology.md) for interpretation.

In [ ]:
if df is not None:
    display(pd.Series(exploratory_revenue_proxy(df, category)))